# Test local chunking b4 moving to Setonix

In [1]:
from pathlib import Path
import warnings
import dask
from dask.distributed import Client
import xarray as xr
import hdf5plugin

dpird_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_DPIRD_utc0_clean/DPIRD_final_stations_utc0.nc")
ecmwf_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_ecmwf_op_clean/2024/02/06.nc")

tmp_drop_path= Path("../.tmp/compressed")

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning
)

## DPIRD dataset

In [2]:
dpird_ds=xr.open_dataset(dpird_path, engine="h5netcdf")
#chunked_dpird_ds= dpird_ds.chunk({'time':3289})
#chunked_dpird_ds

## ECMWF dataset

In [3]:
ecmwf_ds=xr.open_dataset(ecmwf_path, engine="h5netcdf")
#chunked_ecmwf_ds= ecmwf_ds.chunk({"time":2,"step":25})
#chunked_ecmwf_ds

## Chunking and compression
Compression step is in encoding

In [5]:
client= Client(n_workers=4, threads_per_worker=1)
client

d:\Projects\S3-Kerchunk-streamer\virtualiser_venv\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 14151 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:14151/status,
Dashboard: http://127.0.0.1:14151/status,Workers: 4
Total threads: 4,Total memory: 13.86 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:14154,Workers: 0
Dashboard: http://127.0.0.1:14151/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:14173,Total threads: 1
Dashboard: http://127.0.0.1:14174/status,Memory: 3.46 GiB
Nanny: tcp://127.0.0.1:14157,


In [ ]:
out_dir = tmp_drop_path
out_dir.mkdir(parents=True, exist_ok=True)

dpird_out = out_dir / "DPIRD_final_stations_utc0.chunked_zstd.nc"
ecmwf_out = out_dir / "2024_02_06.chunked_zstd.nc"

def build_var_encoding(ds, chunk_dict, complevel=5):
    enc = {}
    for v in ds.data_vars:
        var_chunks = tuple(chunk_dict.get(dim, ds[v].sizes[dim]) for dim in ds[v].dims)

        enc[v] = {
            "zlib": True,
            "complevel": complevel,
            "shuffle": True,
            "chunksizes": var_chunks
        }
    return enc

dpird_chunks = {'time': 3289}
ecmwf_chunks = {"time": 2, "step": 25}
                
dpird_encoding = build_var_encoding(dpird_ds, dpird_chunks)
ecmwf_encoding = build_var_encoding(ecmwf_ds, ecmwf_chunks)

# Dask-lazy netCDF writes
dpird_write = dpird_ds.to_netcdf(
    path=dpird_out,
    engine="h5netcdf",
    format= "NETCDF4",
    encoding=dpird_encoding,
    compute=False,
)

ecmwf_write = ecmwf_ds.to_netcdf(
    path=ecmwf_out,
    engine="h5netcdf",
    format= "NETCDF4",
    encoding=ecmwf_encoding,
    compute=False,
)

# Execute both writes in parallel
dask.compute(dpird_write, ecmwf_write)


ValueError: Compression filter "zstd" is unavailable